# Rebuild Toys & Games 10k and rerun PreBERT

Notebook này tạo **một sample mới gồm đúng 10.000 interactions** từ Amazon Toys & Games 5-core, sau đó chạy Llama preprocessing và đánh giá DeepBERT/PreBERT theo các `k-topic`.

Điểm khác với random sampling 10k dòng: tập mới tiếp tục thỏa **user-item 5-core** (mỗi user và mỗi item có ít nhất 5 reviews). Nhờ vậy validation/test có đủ warm-start interactions và collaborative features có dữ liệu để học. Đây là một dense benchmark thiên về user/item hoạt động cao, vì vậy khi báo cáo cần mô tả sampling protocol và không xem nó là random sample không chệch.

Nguồn: [McAuley Lab Amazon Review Data (2014)](https://mcauleylab.ucsd.edu/public_datasets/data/amazon/index_2014.html), Toys and Games 5-core, 167,597 reviews.

> **MPS:** Sau khi thay đổi environment hoặc kernel, hãy Restart Kernel rồi chạy notebook từ cell đầu. Cell cấu hình sẽ dừng ngay nếu PyTorch không thực sự chạy được trên Apple GPU.

In [1]:
from pathlib import Path
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
os.environ['PREBERT_DEVICE'] = 'mps'

import json
import subprocess
import sys
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'evaluation.py').is_file():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'evaluation.py').is_file(), 'Hãy mở notebook từ repository PreBERT'
if not torch.backends.mps.is_built():
    raise RuntimeError('PyTorch hiện tại không được build với MPS support')
if not torch.backends.mps.is_available():
    raise RuntimeError('MPS không khả dụng; hãy dùng Python arm64 và PyTorch macOS arm64')
MPS_DEVICE = torch.device('mps')
mps_probe = torch.ones(1, device=MPS_DEVICE)
assert mps_probe.device.type == 'mps'

SOURCE_URL = ('https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/'
              'reviews_Toys_and_Games_5.json.gz')
RAW_CACHE = REPO_ROOT / 'data/raw/reviews_Toys_and_Games_5.json.gz'
RAW_SUBSET = REPO_ROOT / 'data/Small_Toys_and_Games_5_dense10k.json'
PROCESSED_SUBSET = REPO_ROOT / 'data/Small_Toys_and_Games_5_dense10k_llama_filtered.json'
DATASET_STEM = PROCESSED_SUBSET.stem
SEMANTIC_OUTPUT = REPO_ROOT / 'exp_llm/semantic_outputs/Small_Toys_and_Games_5_dense10k'

TARGET_SIZE = 10_000
K_CORE = 5
SAMPLE_SEED = 2026
K_TOPICS = [30, 40]
FAST_PREPROCESSING = True  # True: dùng 1B nhanh hơn; False: giữ protocol 3B
LLAMA_MODEL = (
    'meta-llama/Llama-3.2-1B-Instruct'
    if FAST_PREPROCESSING
    else 'meta-llama/Llama-3.2-3B-Instruct'
)

OVERWRITE_SUBSET = False
OVERWRITE_PREPROCESSING = False
RUN_SEMANTIC_QA = True
RUN_TRAINING = True

def run_command(command):
    print('\n$', ' '.join(map(str, command)))
    subprocess.run([str(value) for value in command], cwd=REPO_ROOT, check=True)

print('Repository:', REPO_ROOT)
print('Output dataset:', PROCESSED_SUBSET)
print('PyTorch:', torch.__version__)
print('Active accelerator:', mps_probe.device)

Repository: /Users/trietdang/Triet-Applications/PreBERT
Output dataset: /Users/trietdang/Triet-Applications/PreBERT/data/Small_Toys_and_Games_5_dense10k_llama_filtered.json
PyTorch: 2.8.0
Active accelerator: mps:0


## 1. Download và tạo dense 10k subset

Script tải file `.json.gz` một lần và cache trong `data/raw/`. Nó tìm một vùng item hoạt động cao, lấy maximal 5-core, rồi loại cạnh/node có kiểm soát cho tới đúng 10.000 reviews. Seed cố định giúp tái lập kết quả.

In [2]:
if RAW_SUBSET.exists() and RAW_SUBSET.with_suffix('.sampling_report.json').exists() and not OVERWRITE_SUBSET:
    print('Subset đã tồn tại, bỏ qua:', RAW_SUBSET)
else:
    command = [
        sys.executable, '-m', 'exp_llm', 'build-dataset',
        '--source-url', SOURCE_URL,
        '--raw-cache', RAW_CACHE,
        '--output', RAW_SUBSET,
        '--target-size', TARGET_SIZE,
        '--k-core', K_CORE,
        '--seed', SAMPLE_SEED,
    ]
    if OVERWRITE_SUBSET:
        command.append('--overwrite')
    run_command(command)

Subset đã tồn tại, bỏ qua: /Users/trietdang/Triet-Applications/PreBERT/data/Small_Toys_and_Games_5_dense10k.json


In [3]:
report_path = RAW_SUBSET.with_suffix('.sampling_report.json')
sampling_report = json.loads(report_path.read_text(encoding='utf-8'))
display(pd.DataFrame({
    'Metric': [
        'Reviews', 'Users', 'Items', 'Density',
        'Min reviews/user', 'Median reviews/user',
        'Min reviews/item', 'Median reviews/item'
    ],
    'Value': [
        sampling_report['reviews'], sampling_report['users'], sampling_report['items'],
        sampling_report['density'], sampling_report['min_reviews_per_user'],
        sampling_report['median_reviews_per_user'], sampling_report['min_reviews_per_item'],
        sampling_report['median_reviews_per_item']
    ]
}))
assert sampling_report['reviews'] == TARGET_SIZE
assert sampling_report['min_reviews_per_user'] >= K_CORE
assert sampling_report['min_reviews_per_item'] >= K_CORE
sampling_report['rating_distribution']

,Metric,Value
0,Reviews,10000.000000
1,Users,1190.000000
2,Items,486.000000
3,Density,0.017291
4,Min reviews/user,5.000000
5,Median reviews/user,7.000000
6,Min reviews/item,5.000000
7,Median reviews/item,18.000000


{'1.0': 178, '2.0': 375, '3.0': 1106, '4.0': 2969, '5.0': 5372}

## 2. Llama preprocessing

Cell này gọi đúng pipeline `preprocess_reviews.py`: loại các segment không mang thông tin đánh giá và tạo `filteredReviewText`, `overall_new`. `FAST_PREPROCESSING=False` giữ Llama 3B theo protocol; đặt `True` sẽ dùng Llama 1B nhanh hơn nhưng phải được báo cáo như một preprocessor khác. Nếu Hugging Face trả về 401/403, chạy `huggingface-cli login` một lần trong terminal.

In [4]:
os.environ['PREBERT_DEVICE'] = 'mps'
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

'1'

In [5]:
if PROCESSED_SUBSET.exists() and not OVERWRITE_PREPROCESSING:
    print('Dataset đã preprocessing, bỏ qua:', PROCESSED_SUBSET)
else:
    command = [
        sys.executable, '-m', 'exp_llm', 'preprocess', RAW_SUBSET,
        '--output', PROCESSED_SUBSET,
        '--model', LLAMA_MODEL,
        '--device', 'mps',
        '--batch-size', 16,
        '--review-batch-size', 128,
        '--max-length', 512,
        '--device', 'mps',
    ]
    if OVERWRITE_PREPROCESSING:
        command.append('--overwrite')
    run_command(command)


$ /Users/trietdang/Triet-Applications/PreBERT/.venv/bin/python exp_llm/preprocess_reviews.py /Users/trietdang/Triet-Applications/PreBERT/data/Small_Toys_and_Games_5_dense10k.json --output /Users/trietdang/Triet-Applications/PreBERT/data/Small_Toys_and_Games_5_dense10k_llama_filtered.json --model meta-llama/Llama-3.2-1B-Instruct --device mps --batch-size 16 --review-batch-size 128 --max-length 512 --device mps
LLM review preprocessing
  input: /Users/trietdang/Triet-Applications/PreBERT/data/Small_Toys_and_Games_5_dense10k.json
  output: /Users/trietdang/Triet-Applications/PreBERT/data/Small_Toys_and_Games_5_dense10k_llama_filtered.json
  model: meta-llama/Llama-3.2-1B-Instruct
  rows: 10000
  rows without text: 8
  review segments: 85469
  LLM classifications: 95469
  adjust ratings: True
  requested device: mps
  active device: mps:0
Processed 128/10000 | 4.12 reviews/s | ETA 39.9 min
Processed 256/10000 | 4.03 reviews/s | ETA 40.3 min
Processed 384/10000 | 4.14 reviews/s | ETA 38.7 

In [6]:
processed = pd.read_json(PROCESSED_SUBSET, lines=True)
assert len(processed) == TARGET_SIZE
assert {'filteredReviewText', 'overall_new'}.issubset(processed.columns)
changed = processed['reviewText'].fillna('').str.strip().ne(
    processed['filteredReviewText'].fillna('').str.strip()
)
rating_changed = processed['overall'].ne(processed['overall_new'])
pd.Series({
    'rows': len(processed),
    'changed_reviews': int(changed.sum()),
    'changed_review_rate': float(changed.mean()),
    'adjusted_ratings': int(rating_changed.sum()),
    'empty_filtered_reviews': int(processed['filteredReviewText'].fillna('').str.strip().eq('').sum()),
})

rows                      10000.0000
changed_reviews            8184.0000
changed_review_rate           0.8184
adjusted_ratings           1022.0000
empty_filtered_reviews        8.0000
dtype: float64

## 3. Semantic-retention QA (khuyến nghị)

Chỉ đánh giá những review thực sự bị thay đổi. Kết quả được lưu riêng để báo cáo rằng preprocessing đã rút gọn text nhưng vẫn giữ nội dung ngữ nghĩa chính.

In [7]:
if RUN_SEMANTIC_QA:
    run_command([
        sys.executable, '-m', 'exp_llm', 'semantic', PROCESSED_SUBSET,
        '--output-dir', SEMANTIC_OUTPUT,
        '--only-changed',
        '--device', 'mps',
        '--batch-size', 32,
    ])
else:
    print('Semantic QA is disabled')


$ /Users/trietdang/Triet-Applications/PreBERT/.venv/bin/python exp_llm/evaluate_semantic_retention.py /Users/trietdang/Triet-Applications/PreBERT/data/Small_Toys_and_Games_5_dense10k_llama_filtered.json --output-dir /Users/trietdang/Triet-Applications/PreBERT/exp_llm/semantic_outputs/Small_Toys_and_Games_5_dense10k --only-changed --device mps --batch-size 32
Input rows: 10000
Changed rows: 6538
Rows evaluated: 6538
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: mps
{
  "input": "/Users/trietdang/Triet-Applications/PreBERT/data/Small_Toys_and_Games_5_dense10k_llama_filtered.json",
  "model": "sentence-transformers/all-MiniLM-L6-v2",
  "device": "mps",
  "processed_field": "filteredReviewText",
  "selection": "changed_only",
  "threshold": 0.8,
  "input_rows": 10000,
  "changed_rows": 6538,
  "rows_evaluated": 6538,
  "mean_retained_word_ratio": 0.4248420722452047,
  "mean_removed_word_ratio": 0.5751579277547954,
  "empty_processed_rate": 0.0,
  "references": {
    "rev

## 4. Rerun DeepBERT/PreBERT

`evaluation.py` sẽ chạy riêng dataset mới cho từng `k-topic`, tự xóa feature/checkpoint tạm giữa các run, split warm-start, fit preprocessing/model trên train, chọn checkpoint bằng validation và chỉ báo cáo test ở cuối. Kết quả nằm trong `results/`.

In [8]:
if RUN_TRAINING:
    run_command([
        sys.executable, 'evaluation.py',
        '--datasets', DATASET_STEM,
        '--topics', *K_TOPICS,
        '--batch-size', 32,
        '--epochs', 100,
        '--num-words', 200,
        '--seed', 42,
    ])
else:
    print('Training is disabled')


$ /Users/trietdang/Triet-Applications/PreBERT/.venv/bin/python evaluation.py --datasets Small_Toys_and_Games_5_dense10k_llama_filtered --topics 30 40 --batch-size 32 --epochs 100 --num-words 200 --seed 42


/Users/trietdang/Triet-Applications/PreBERT/init.py:79: RuntimeWarning: Stanford dependency parsing is disabled: Java Runtime is unavailable (Command '['java', '-version']' returned non-zero exit status 1.). Continuing with topic-word sentiment only. Install a JDK to enable dependency neighbours.
  dep_parser = DependencyParser(MODEL_PATH, PARSER_PATH)


Using device: mps

Dataset: Small_Toys_and_Games_5_dense10k_llama_filtered | k-topic: 30
Removed 14 generated artifact(s) from the previous run
Split sizes: train=7000, valid=1000, test=2000 (100% warm-start validation/test)
Method:  DeepBERT
Using device: mps


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/10: 100%|██████████| 875/875 [10:17<00:00,  1.42batch/s, Loss=1.05]


BERT epoch 1: train loss=1.050783, valid loss=0.976586


...

Birch
data_train_size:  7000


100%|██████████| 486/486 [08:13<00:00,  1.01s/it]


SVD iteration 10: train RMSE=4.296739, valid RMSE=4.278463
SVD iteration 20: train RMSE=4.283084, valid RMSE=4.276460
SVD iteration 30: train RMSE=4.216139, valid RMSE=4.255829
SVD iteration 40: train RMSE=3.912727, valid RMSE=4.123952
SVD iteration 50: train RMSE=2.833137, valid RMSE=3.467182
SVD iteration 60: train RMSE=1.079579, valid RMSE=2.236893
SVD iteration 70: train RMSE=0.522981, valid RMSE=1.858920
SVD iteration 80: train RMSE=0.442049, valid RMSE=1.817035
SVD iteration 90: train RMSE=0.398080, valid RMSE=1.811049
SVD iteration 100: train RMSE=0.363306, valid RMSE=1.809975
SVD iteration 110: train RMSE=0.333923, valid RMSE=1.810410
SVD iteration 120: train RMSE=0.308406, valid RMSE=1.811635
SVD iteration 130: train RMSE=0.285876, valid RMSE=1.813321
SVD iteration 140: train RMSE=0.265755, valid RMSE=1.815265
SVD iteration 150: train RMSE=0.247638, valid RMSE=1.817332
SVD iteration 160: train RMSE=0.231228, valid RMSE=1.819427
SVD iteration 170: train RMSE=0.216298, valid RMS

100%|██████████| 63/63 [00:00<00:00, 604.51it/s]


rsme raw:  0.7736190932615149
MAE:  0.5769682252407073
Đã lưu danh sách giá trị vào tệp 'results/DeepBERT_Small_Toys_and_Games_5_dense10k_llama_filtered_factors40.xlsx' thành công.


In [9]:
result_rows = []
for topic in K_TOPICS:
    path = REPO_ROOT / f'results/DeepBERT_{DATASET_STEM}_factors{topic}.xlsx'
    if path.is_file():
        row = pd.read_excel(path).iloc[-1].to_dict()
        result_rows.append({'Dataset': DATASET_STEM, 'k-topic': topic, **row})
    else:
        print('Chưa có result:', path)
pd.DataFrame(result_rows)

,Dataset,k-topic,Accuracy,RMSE Test,MAE Test
0,Small_Toys_and_Games_5_dense10k_llama_filtered,30,0.8465,0.7737,0.5765
1,Small_Toys_and_Games_5_dense10k_llama_filtered,40,0.8465,0.7736,0.5770
